# 4 Shapes Detection and Recognition

This notebook demonstrates geometric shape detection on the 4shapes.jpg image.

**Image:** 4shapes.jpg  
**Techniques Used:**
- Contour detection
- Vertex counting
- Shape approximation
- Morphological operations

## 1. Import Libraries

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Libraries imported successfully!")

## 2. Define Shape Detection Function

In [ ]:
def detect_shape(contour):
    """Detect shape based on number of vertices"""
    shape = "unidentified"
    peri = cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, 0.04 * peri, True)
    vertices = len(approx)
    
    if vertices == 3:
        shape = "triangle"
    elif vertices == 4:
        x, y, w, h = cv2.boundingRect(approx)
        aspect_ratio = w / float(h)
        shape = "square" if 0.95 <= aspect_ratio <= 1.05 else "rectangle"
    elif vertices == 5:
        shape = "pentagon"
    elif vertices == 6:
        shape = "hexagon"
    elif vertices > 6:
        shape = "circle"
    
    return shape, vertices

## 3. Load and Display Image

In [ ]:
# Load image
image_path = "../images/4shapes.jpg"
image = cv2.imread(image_path)

if image is None:
    print(f"Error: Could not load image from {image_path}")
else:
    original = image.copy()
    print(f"Image loaded successfully!")
    print(f"Image shape: {image.shape}")
    
    # Display original image
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    plt.title('Original Image - 4 Shapes', fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()

## 4. Image Preprocessing

In [ ]:
# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Apply Gaussian blur
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

# Binary threshold
_, thresh = cv2.threshold(blurred, 127, 255, cv2.THRESH_BINARY)

# Morphological operations
kernel = np.ones((5, 5), np.uint8)
morph = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

# Edge detection
edges = cv2.Canny(morph, 50, 150)

# Display preprocessing steps
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Image Preprocessing Steps - 4 Shapes', fontsize=16, fontweight='bold')

axes[0, 0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

axes[0, 1].imshow(gray, cmap='gray')
axes[0, 1].set_title('Grayscale')
axes[0, 1].axis('off')

axes[0, 2].imshow(blurred, cmap='gray')
axes[0, 2].set_title('Gaussian Blur')
axes[0, 2].axis('off')

axes[1, 0].imshow(thresh, cmap='gray')
axes[1, 0].set_title('Binary Threshold')
axes[1, 0].axis('off')

axes[1, 1].imshow(morph, cmap='gray')
axes[1, 1].set_title('Morphological Closing')
axes[1, 1].axis('off')

axes[1, 2].imshow(edges, cmap='gray')
axes[1, 2].set_title('Edge Detection')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## 5. Contour Detection and Shape Recognition

In [ ]:
# Find contours
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Create result image
result = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
shape_counts = {}

print(f"Detected {len(contours)} contours\n")
print(f"{'Shape':<15} {'Vertices':<10} {'Area':<12} {'Perimeter':<12}")
print("-" * 60)

for i, contour in enumerate(contours):
    area = cv2.contourArea(contour)
    if area > 500:  # Filter small contours
        shape, vertices = detect_shape(contour)
        perimeter = cv2.arcLength(contour, True)
        
        # Count shapes
        shape_counts[shape] = shape_counts.get(shape, 0) + 1
        
        # Draw contour and label
        cv2.drawContours(result, [contour], -1, (0, 255, 0), 3)
        M = cv2.moments(contour)
        if M["m00"] != 0:
            cX = int(M["m10"] / M["m00"])
            cY = int(M["m01"] / M["m00"])
            cv2.putText(result, shape, (cX - 30, cY), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        
        print(f"{shape:<15} {vertices:<10} {area:<12.2f} {perimeter:<12.2f}")

print("\n" + "="*60)
print("SHAPE SUMMARY:")
for shape, count in shape_counts.items():
    print(f"  {shape.capitalize()}: {count}")
print("="*60)

## 6. Display Results

In [ ]:
# Display final result
plt.figure(figsize=(12, 8))
plt.imshow(result)
plt.title(f'Detected Shapes - 4 Shapes Image', fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

## 7. Side-by-Side Comparison

In [ ]:
# Compare original and result
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Shape Detection Results - 4 Shapes', fontsize=16, fontweight='bold')

axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(result)
axes[1].set_title('Detected Shapes')
axes[1].axis('off')

plt.tight_layout()
plt.show()